# Brain Neuron Digital Twin (Google Colab)

Three parts, run top to bottom (**Runtime → Run all**):

1. **Simulator + dataset**: a leaky integrate-and-fire neuron generates labelled data (`neuron_lif_dataset.csv`).
2. **ML prediction**: a neural network learns to predict firing rate from synaptic input.
3. **Digital twin**: a 3D neuron, an animated signal travelling along it, and an interactive prediction panel.

*Optional:* upload `neuron_lif_dataset.csv` to the Colab file sidebar and it will be loaded instead of regenerated.

## 1. Simulator and dataset

In [ ]:
import numpy as np, pandas as pd

# ---- Leaky integrate-and-fire (LIF) neuron: the "physics" of the digital twin ----
W_E, W_I   = 1.0, 1.5     # mV kick per excitatory / inhibitory synaptic event
TAU        = 20.0         # membrane time constant (ms)
V_REST, V_TH = -70.0, -50.0
REFRAC     = 4            # refractory period (ms)

def simulate_lif(n_exc, n_inh, rate, T_ms=500, seed=None, record=False):
    """Vectorised LIF simulation. Inputs can be arrays -> one neuron run per element.
    Returns firing rate (Hz) and, if record=True, the membrane-potential trace (T_ms x N)."""
    rng = np.random.default_rng(seed)
    n_exc, n_inh, rate = [np.atleast_1d(np.asarray(a, float)) for a in (n_exc, n_inh, rate)]
    n = len(n_exc)
    lam_e, lam_i = n_exc * rate / 1000, n_inh * rate / 1000   # expected events per ms
    V = np.full(n, V_REST); ref = np.zeros(n, int); spikes = np.zeros(n, int)
    trace = np.empty((T_ms, n)) if record else None
    for t in range(T_ms):
        ne, ni = rng.poisson(lam_e), rng.poisson(lam_i)
        in_ref = ref > 0
        V_new = np.maximum(V - (V - V_REST) / TAU + W_E * ne - W_I * ni, -85.0)
        V = np.where(in_ref, V_REST, V_new)
        ref = np.where(in_ref, ref - 1, ref)
        fired = (~in_ref) & (V >= V_TH)
        spikes += fired
        if record: trace[t] = np.where(fired, -10.0, V)
        V = np.where(fired, V_REST, V)
        ref = np.where(fired, REFRAC, ref)
    out = spikes * 1000 / T_ms
    return (out, trace) if record else out

def make_dataset(n=5000, T_ms=500, seed=42):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "n_exc":   rng.uniform(0, 80, n).round(1),   # number of excitatory synapses
        "n_inh":   rng.uniform(0, 40, n).round(1),   # number of inhibitory synapses
        "rate_hz": rng.uniform(1, 50, n).round(1),   # input rate per synapse (Hz)
    })
    df["net_drive"] = np.maximum(0, df.n_exc - 1.5 * df.n_inh) * df.rate_hz / 4000
    df["firing_rate_hz"] = simulate_lif(df.n_exc, df.n_inh, df.rate_hz, T_ms, seed=seed + 1)
    df["spiked"] = (df.firing_rate_hz > 0).astype(int)
    return df

In [ ]:
import os
CSV = "neuron_lif_dataset.csv"
if os.path.exists(CSV):
    df = pd.read_csv(CSV); print("Loaded", CSV)
else:
    df = make_dataset(n=5000); df.to_csv(CSV, index=False); print("Generated", CSV)
print(df.shape)
df.head()

## 2. Train the prediction model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

FEATURES = ["n_exc", "n_inh", "rate_hz", "net_drive"]
X_tr, X_te, y_tr, y_te = train_test_split(df[FEATURES], df.firing_rate_hz, test_size=0.2, random_state=0)

models = {
    "MLP (neural net)": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=0)),
    "Random forest":    RandomForestRegressor(200, random_state=0, n_jobs=-1),
}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = np.maximum(0, m.predict(X_te))
    print(f"{name:18s}  R2 = {r2_score(y_te, p):.3f}   MAE = {mean_absolute_error(y_te, p):.2f} Hz")

mlp = models["MLP (neural net)"]
pred = np.maximum(0, mlp.predict(X_te))
plt.figure(figsize=(5, 5))
plt.scatter(y_te, pred, s=6, alpha=.4)
plt.plot([0, y_te.max()], [0, y_te.max()], "r--", lw=1)
plt.xlabel("Simulated firing rate (Hz)"); plt.ylabel("Predicted (Hz)"); plt.title("Held-out test set")
plt.show()

## 3a. Grow the 3D neuron

In [ ]:
import plotly.graph_objects as go

rng = np.random.default_rng(11)
segs, tip_len = [], []          # each seg: dict(a, b, type, c0, c1, leaves)
ax_max = 0.0

def perturb(d, ang):
    v = np.cross(rng.random(3) - .5, d); v /= np.linalg.norm(v)
    out = d*np.cos(ang) + np.cross(v, d)*np.sin(ang) + v*np.dot(v, d)*(1 - np.cos(ang))
    return out / np.linalg.norm(out)

def dendrite(p, d, depth, dist):
    L = 13 * .74**depth * (.8 + rng.random()*.4); b = p + d*L
    s = dict(a=p, b=b, type="d", c0=dist, c1=dist+L, leaves=[]); segs.append(s)
    if depth >= 4:
        tip_len.append(dist+L); s["leaves"] = [len(tip_len)-1]; return s["leaves"]
    ids = []
    for _ in range(3 if rng.random() < .2 else 2):
        ids += dendrite(b, perturb(d, .45 + rng.random()*.4), depth+1, dist+L)
    s["leaves"] = ids; return ids

def axon(p, d, n, dist, seg_len, branch):
    global ax_max
    for i in range(n):
        d = perturb(d, .12 + rng.random()*.1); L = seg_len*(.9 + rng.random()*.2); b = p + d*L
        segs.append(dict(a=p, b=b, type="a", c0=dist, c1=dist+L, leaves=[]))
        dist += L; p = b; ax_max = max(ax_max, dist)
        if branch and i in (3, 6): axon(p, perturb(d, .9), 3, dist, 6, False)
    return p, d, dist

SOMA_R = 4.5
for i in range(7):
    th = 2*np.pi*i/7 + rng.random()*.5; y = -.15 + rng.random(); h = np.sqrt(1 - y*y)
    d = np.array([np.cos(th)*h, y, np.sin(th)*h]); d /= np.linalg.norm(d)
    dendrite(d*(SOMA_R-.5), d, 0, 0.0)
p, d, dist = axon(np.array([0, -SOMA_R+.5, 0]), np.array([0, -1.0, 0]), 11, 0.0, 7.5, True)
for _ in range(3): axon(p, perturb(d, .5 + rng.random()*.4), 3, dist, 5, False)

print(f"{len(segs)} segments, {len(tip_len)} dendritic tips, axon length {ax_max:.0f}")

def lines(kind):
    xs, ys, zs = [], [], []
    for s in segs:
        if s["type"] == kind:
            for k, arr in enumerate((xs, ys, zs)): arr += [s["a"][k], s["b"][k], None]
    return xs, ys, zs

# sample points along each segment: these carry the animated signal
PTS, PC, PTYPE, PMASK = [], [], [], []
for s in segs:
    for f in (.125, .375, .625, .875):
        PTS.append(s["a"] + f*(s["b"] - s["a"])); PC.append(s["c0"] + f*(s["c1"] - s["c0"])); PTYPE.append(s["type"])
        m = np.zeros(len(tip_len), bool); m[s["leaves"]] = True; PMASK.append(m)
PTS, PC, PTYPE, PMASK = np.array(PTS), np.array(PC), np.array(PTYPE), np.array(PMASK)

u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi:16j]
soma = go.Surface(x=SOMA_R*np.cos(u)*np.sin(v), y=SOMA_R*np.cos(v), z=SOMA_R*np.sin(u)*np.sin(v),
                  colorscale=[[0, "#b093d6"], [1, "#b093d6"]], showscale=False, hoverinfo="skip")
dx, dy, dz = lines("d"); ax_, ay, az = lines("a")
base = [go.Scatter3d(x=dx, y=dy, z=dz, mode="lines", line=dict(color="#8f6fb5", width=3), hoverinfo="skip", name="Dendrites"),
        go.Scatter3d(x=ax_, y=ay, z=az, mode="lines", line=dict(color="#6a93c6", width=4), hoverinfo="skip", name="Axon"),
        soma]
LAYOUT = dict(template="plotly_dark", height=650, margin=dict(l=0, r=0, t=40, b=0), paper_bgcolor="#1b1632",
              scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False, aspectmode="data",
                         bgcolor="#1b1632", camera=dict(eye=dict(x=1.5, y=.2, z=.3))), legend=dict(x=0, y=1))
go.Figure(base, layout=dict(LAYOUT, title="Procedurally grown neuron (drag to rotate)")).show()

## 3b. Animate a signal: inputs travel to the soma, then a spike travels down the axon

In [ ]:
g = lambda x: np.exp(-x**2)
n_frames, split = 90, 45
sel = np.random.default_rng(3).choice(len(tip_len), 10, replace=False)      # tips receiving input
start = np.random.default_rng(4).integers(0, 14, len(sel))
tip_arr = np.array(tip_len); d_speed = tip_arr.max() / 30; a_speed = ax_max / 38
is_d, is_a = PTYPE == "d", PTYPE == "a"

def frame_data(t):
    e = np.zeros(len(PC)); sp = np.zeros(len(PC))
    for k, tip in enumerate(sel):
        c = tip_arr[tip] - d_speed * max(0, t - start[k])
        if t >= start[k] and c > -2:
            e = np.maximum(e, np.where(is_d & PMASK[:, tip], g((PC - c)/6), 0))
    if t >= split:
        c = a_speed * (t - split)
        sp = np.where(is_a, g((PC - c)/7), 0)
    act = np.maximum(e, sp)
    colour = np.where(sp > e, 2, 1)
    size = np.where(act > .08, 2 + 9*act, 0.1)
    flash = 28 * np.exp(-((t - split)/3)**2) + 0.1
    return colour, size, flash

c0, s0, f0 = frame_data(0)
pts = go.Scatter3d(x=PTS[:, 0], y=PTS[:, 1], z=PTS[:, 2], mode="markers", hoverinfo="skip", showlegend=False,
                   marker=dict(color=c0, size=s0, cmin=1, cmax=2, opacity=.95,
                               colorscale=[[0, "#ff8fbf"], [.5, "#ff8fbf"], [.5, "#ffd36b"], [1, "#ffd36b"]]))
fl = go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", hoverinfo="skip", showlegend=False,
                  marker=dict(color="#ffd36b", size=f0, opacity=.5))
frames = []
for t in range(n_frames):
    c, s, f = frame_data(t)
    frames.append(go.Frame(name=str(t), traces=[3, 4],
        data=[go.Scatter3d(marker=dict(color=c, size=s, cmin=1, cmax=2)), go.Scatter3d(marker=dict(size=f))]))
fig = go.Figure(base + [pts, fl], frames=frames, layout=dict(LAYOUT, title="Pink: excitatory input.  Amber: spike.",
    updatemenus=[dict(type="buttons", x=.02, y=.02, buttons=[
        dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True, mode="immediate")]),
        dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0), mode="immediate")])])]))
fig.show()

## 3c. Interactive prediction panel

Move the sliders: the trained network predicts the firing rate instantly, and a live LIF simulation checks it.

In [ ]:
import ipywidgets as w
from IPython.display import display

sl = lambda d, lo, hi, val: w.IntSlider(value=val, min=lo, max=hi, description=d, style=dict(description_width="130px"),
                                        layout=w.Layout(width="440px"), continuous_update=False)
s_e, s_i, s_r = sl("Excitatory synapses", 0, 80, 55), sl("Inhibitory synapses", 0, 40, 8), sl("Input rate (Hz)", 1, 50, 26)
out = w.Output()

def update(*_):
    ne, ni, r = s_e.value, s_i.value, s_r.value
    x = pd.DataFrame([[ne, ni, r, max(0, ne - 1.5*ni) * r / 4000]], columns=FEATURES)
    pr = max(0.0, float(mlp.predict(x)[0]))
    sim, tr = simulate_lif(ne, ni, r, T_ms=1000, record=True)
    with out:
        out.clear_output(wait=True)
        print(f"Predicted firing rate : {pr:6.1f} Hz")
        print(f"Simulated (1 s run)   : {sim[0]:6.1f} Hz")
        print(f"Spike chance, next 100 ms (predicted): {100*(1 - np.exp(-pr*.1)):.0f}%")
        plt.figure(figsize=(8, 2.6)); plt.plot(tr[:400, 0], lw=1)
        plt.axhline(V_TH, color="gray", ls="--", lw=1); plt.ylim(-90, 0)
        plt.xlabel("ms"); plt.ylabel("Membrane potential (mV)"); plt.title("First 400 ms of the simulated run (dashed = threshold)")
        plt.tight_layout(); plt.show()

for s in (s_e, s_i, s_r): s.observe(update, "value")
display(w.VBox([s_e, s_i, s_r, out])); update()

## 4. Save / download

The dataset is already saved as `neuron_lif_dataset.csv` in the Colab file sidebar. Run the next cell to download it and the trained model.

In [ ]:
import joblib
joblib.dump(mlp, "neuron_mlp.joblib")
try:
    from google.colab import files
    files.download("neuron_lif_dataset.csv"); files.download("neuron_mlp.joblib")
except ImportError:
    print("Saved neuron_lif_dataset.csv and neuron_mlp.joblib locally.")